# Role Dimension Loader

Maintains the `warehouse.dim_role` dimension table with incremental refresh capability.

## Purpose
Track canonical job roles from governed metadata taxonomy for consistent role classification.

## Key Features
* Stable surrogate keys for roles (auto-increment)
* Role families and sector associations
* Seniority levels and career progression
* Manager/executive flags for filtering
* Role aliases for fuzzy matching
* SCD Type 1 (overwrite on change)
* Idempotent: safe to re-run

## Architecture
**Source**: Unity Catalog taxonomy tables (`workspace.metadata.taxonomy_role_canonical`, `taxonomy_role_families`, `taxonomy_sectors`)  
**Target**: `workspace.warehouse.dim_role`  
**Metadata**: `workspace.metadata.dim_role_refresh_log`  
**Mode**: Incremental (merge new/updated roles)

## Batch Processing
* Tracks refresh history in metadata table
* Auto-assigns surrogate keys to new roles
* Updates existing roles with latest taxonomy
* Validates hierarchy and data quality after refresh

In [0]:
dbutils.widgets.dropdown("force_full_refresh", "false", ["true", "false"], "Force Full Refresh")
dbutils.widgets.text("metadata_path", "/Workspace/Users/aaryan.shrivastav1403@gmail.com/LMIP/metadata", "Metadata Path")

FORCE_FULL_REFRESH = dbutils.widgets.get("force_full_refresh") == "true"
METADATA_PATH = dbutils.widgets.get("metadata_path")

In [0]:
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType, BooleanType, LongType
import json

CATALOG = "workspace"
WAREHOUSE_SCHEMA = f"{CATALOG}.warehouse"
METADATA_SCHEMA = f"{CATALOG}.metadata"

TARGET_TABLE = f"{WAREHOUSE_SCHEMA}.dim_role"
METADATA_TABLE = f"{METADATA_SCHEMA}.dim_role_refresh_log"

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
run_timestamp = datetime.now()

print(f"Run ID: {run_id}")
print(f"Metadata path: {METADATA_PATH}")
print(f"Force full refresh: {FORCE_FULL_REFRESH}")

In [0]:
%sql
-- Create role dimension table if not exists
CREATE TABLE IF NOT EXISTS workspace.warehouse.dim_role (
  role_sk BIGINT NOT NULL COMMENT 'Surrogate key for role',
  canonical_role_id STRING NOT NULL COMMENT 'Natural key from taxonomy (role_key)',
  role_name STRING NOT NULL COMMENT 'Canonical role name',
  family_sk BIGINT COMMENT 'Surrogate key to role family dimension',
  family_key STRING COMMENT 'Natural key for role family',
  role_family STRING COMMENT 'Role family name',
  sector_sk BIGINT COMMENT 'Surrogate key to sector dimension',
  sector_key STRING COMMENT 'Natural key for sector',
  seniority_level STRING NOT NULL COMMENT 'Seniority level (entry, mid, senior, executive)',
  role_category STRING NOT NULL COMMENT 'Role category classification',
  role_level INT NOT NULL COMMENT 'Career level numeric',
  role_description STRING COMMENT 'Role description from taxonomy',
  aliases ARRAY<STRING> COMMENT 'Role aliases for matching',
  is_manager BOOLEAN NOT NULL COMMENT 'Is manager role',
  is_executive BOOLEAN NOT NULL COMMENT 'Is executive role',
  is_active BOOLEAN NOT NULL COMMENT 'Is role active',
  created_at TIMESTAMP NOT NULL COMMENT 'Record creation timestamp',
  updated_at TIMESTAMP NOT NULL COMMENT 'Last update timestamp',
  taxonomy_updated_at TIMESTAMP COMMENT 'Timestamp from source taxonomy table',
  CONSTRAINT pk_dim_role PRIMARY KEY (role_sk)
)
USING DELTA
COMMENT 'Canonical role dimension from governed taxonomy tables';

-- Create metadata tracking table
CREATE TABLE IF NOT EXISTS workspace.metadata.dim_role_refresh_log (
  run_id STRING,
  roles_extracted INT,
  roles_inserted INT,
  roles_updated INT,
  force_full_refresh BOOLEAN,
  processed_at TIMESTAMP,
  status STRING,
  error_message STRING
)
USING DELTA
COMMENT 'Tracks role dimension refresh history';

In [0]:
print("Loading role taxonomy from metadata tables...", end=" ")

# Load from taxonomy tables
roles_df = spark.table(f"{METADATA_SCHEMA}.taxonomy_role_canonical")
role_families_df = spark.table(f"{METADATA_SCHEMA}.taxonomy_role_families")
sectors_df = spark.table(f"{METADATA_SCHEMA}.taxonomy_sectors")

# Transform for warehouse dimension with proper surrogate key lookups
role_extract_df = roles_df.alias("r") \
    .join(
        role_families_df.alias("f"),
        F.col("r.family_key") == F.col("f.family_key"),
        "left"
    ) \
    .join(
        sectors_df.alias("s"),
        F.col("f.sector_key") == F.col("s.sector_key"),
        "left"
    ) \
    .select(
        F.col("r.role_key").alias("canonical_role_id"),
        F.col("r.canonical_role").alias("canonical_role_name"),
        F.lit(None).cast("bigint").alias("family_sk"),  # Will be assigned during merge
        F.col("r.family_key").alias("family_key"),
        F.col("f.family_name").alias("role_family"),
        F.lit(None).cast("bigint").alias("sector_sk"),  # Will be assigned during merge
        F.col("f.sector_key").alias("sector_key"),
        F.col("r.seniority").alias("seniority_level"),
        F.when(F.col("r.seniority") == "senior", "senior").otherwise("individual_contributor").alias("role_category"),
        F.lit(1).alias("role_level"),
        F.col("r.role_description").alias("role_description"),
        F.split(F.col("r.aliases"), "\\|").alias("aliases"),
        F.when(F.col("r.seniority").isin(["senior", "executive"]), True).otherwise(False).alias("is_manager"),
        F.when(F.col("r.seniority") == "executive", True).otherwise(False).alias("is_executive"),
        F.lit(True).alias("is_active"),
        F.col("r.updated_at").alias("taxonomy_updated_at")
    )

roles_count = role_extract_df.count()
print(f"✓ Loaded {roles_count} roles from taxonomy tables")

# Get current max surrogate key
max_sk_result = spark.sql(f"SELECT COALESCE(MAX(role_sk), 0) as max_sk FROM {TARGET_TABLE}").collect()
max_sk = max_sk_result[0]['max_sk']

print(f"Current max surrogate key: {max_sk}")

In [0]:
# Define metadata schema
metadata_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("roles_extracted", IntegerType(), True),
    StructField("roles_inserted", IntegerType(), True),
    StructField("roles_updated", IntegerType(), True),
    StructField("force_full_refresh", BooleanType(), True),
    StructField("processed_at", TimestampType(), True),
    StructField("status", StringType(), True),
    StructField("error_message", StringType(), True)
])

try:
    print(f"Processing roles into {TARGET_TABLE}...", end=" ")
    
    # Check if table schema matches expected schema FIRST
    existing_cols = [row.col_name for row in spark.sql(f"DESCRIBE {TARGET_TABLE}").collect()]
    expected_cols = ['role_sk', 'canonical_role_id', 'role_name', 'family_sk', 'family_key', 'role_family', 'sector_sk', 'sector_key', 'seniority_level', 'role_category', 'role_level', 'role_description', 'aliases', 'is_manager', 'is_executive', 'is_active', 'created_at', 'updated_at', 'taxonomy_updated_at']
    schema_matches = set(existing_cols) == set(expected_cols)
    
    # Only query existing roles if schema matches
    if schema_matches:
        existing_roles = spark.sql(f"SELECT canonical_role_id, role_sk FROM {TARGET_TABLE}")
        
        # Join to assign keys (existing or new)
        from pyspark.sql.window import Window
        
        roles_with_keys = role_extract_df.alias("r").join(
            existing_roles.alias("e"),
            F.col("r.canonical_role_id") == F.col("e.canonical_role_id"),
            "left"
        )
        
        # Assign surrogate keys
        window_spec = Window.orderBy("r.canonical_role_id")
        
        roles_final = roles_with_keys.withColumn(
            "role_sk",
            F.coalesce(F.col("e.role_sk"), F.lit(max_sk) + F.row_number().over(window_spec))
        ).withColumn(
            "created_at",
            F.when(F.col("e.role_sk").isNull(), F.lit(run_timestamp)).otherwise(F.lit(run_timestamp))
        ).withColumn(
            "updated_at",
            F.lit(run_timestamp)
        ).select(
            F.col("role_sk").cast(LongType()),
            F.col("r.canonical_role_id").alias("canonical_role_id"),
            F.col("r.canonical_role_name").alias("role_name"),
            F.col("r.family_sk").alias("family_sk"),
            F.col("r.family_key").alias("family_key"),
            F.col("r.role_family").alias("role_family"),
            F.col("r.sector_sk").alias("sector_sk"),
            F.col("r.sector_key").alias("sector_key"),
            F.col("r.seniority_level").alias("seniority_level"),
            F.col("r.role_category").alias("role_category"),
            F.col("r.role_level").alias("role_level"),
            F.col("r.role_description").alias("role_description"),
            F.col("r.aliases").alias("aliases"),
            F.col("r.is_manager").alias("is_manager"),
            F.col("r.is_executive").alias("is_executive"),
            F.col("r.is_active").alias("is_active"),
            "created_at",
            "updated_at",
            F.col("r.taxonomy_updated_at").alias("taxonomy_updated_at")
        )
    else:
        # Schema mismatch: assign keys without joining to existing table
        from pyspark.sql.window import Window
        window_spec = Window.orderBy("canonical_role_id")
        
        roles_final = role_extract_df.withColumn(
            "role_sk",
            (F.lit(max_sk) + F.row_number().over(window_spec)).cast(LongType())
        ).withColumn(
            "created_at",
            F.lit(run_timestamp)
        ).withColumn(
            "updated_at",
            F.lit(run_timestamp)
        ).select(
            "role_sk",
            "canonical_role_id",
            F.col("canonical_role_name").alias("role_name"),
            "family_sk",
            "family_key",
            "role_family",
            "sector_sk",
            "sector_key",
            "seniority_level",
            "role_category",
            "role_level",
            "role_description",
            "aliases",
            "is_manager",
            "is_executive",
            "is_active",
            "created_at",
            "updated_at",
            "taxonomy_updated_at"
        )
    
    # Create temp view for merge
    roles_final.createOrReplaceTempView("roles_to_merge")
    
    if FORCE_FULL_REFRESH or not schema_matches:
        # Full refresh: drop and recreate with new schema
        if not schema_matches:
            print(f"Schema mismatch detected. Performing full refresh...")
            spark.sql(f"DROP TABLE IF EXISTS {TARGET_TABLE}")
            spark.sql(f"""
            CREATE TABLE {TARGET_TABLE} (
              role_sk BIGINT NOT NULL COMMENT 'Surrogate key for role',
              canonical_role_id STRING NOT NULL COMMENT 'Natural key from taxonomy (role_key)',
              role_name STRING NOT NULL COMMENT 'Canonical role name',
              family_sk BIGINT COMMENT 'Surrogate key to role family dimension',
              family_key STRING COMMENT 'Natural key for role family',
              role_family STRING COMMENT 'Role family name',
              sector_sk BIGINT COMMENT 'Surrogate key to sector dimension',
              sector_key STRING COMMENT 'Natural key for sector',
              seniority_level STRING NOT NULL COMMENT 'Seniority level (entry, mid, senior, executive)',
              role_category STRING NOT NULL COMMENT 'Role category classification',
              role_level INT NOT NULL COMMENT 'Career level numeric',
              role_description STRING COMMENT 'Role description from taxonomy',
              aliases ARRAY<STRING> COMMENT 'Role aliases for matching',
              is_manager BOOLEAN NOT NULL COMMENT 'Is manager role',
              is_executive BOOLEAN NOT NULL COMMENT 'Is executive role',
              is_active BOOLEAN NOT NULL COMMENT 'Is role active',
              created_at TIMESTAMP NOT NULL COMMENT 'Record creation timestamp',
              updated_at TIMESTAMP NOT NULL COMMENT 'Last update timestamp',
              taxonomy_updated_at TIMESTAMP COMMENT 'Timestamp from source taxonomy table',
              CONSTRAINT pk_dim_role PRIMARY KEY (role_sk)
            )
            USING DELTA
            COMMENT 'Canonical role dimension from governed taxonomy tables'
            """)
        else:
            spark.sql(f"TRUNCATE TABLE {TARGET_TABLE}")
        
        roles_final.write.format("delta").mode("append").saveAsTable(TARGET_TABLE)
        roles_inserted = roles_final.count()
        roles_updated = 0
        print(f"✓ Full refresh: {roles_inserted} roles inserted")
    else:
        # Incremental: merge
        merge_sql = f"""
        MERGE INTO {TARGET_TABLE} target
        USING roles_to_merge source
        ON target.canonical_role_id = source.canonical_role_id
        WHEN MATCHED THEN UPDATE SET
            target.role_name = source.role_name,
            target.family_sk = source.family_sk,
            target.family_key = source.family_key,
            target.role_family = source.role_family,
            target.sector_sk = source.sector_sk,
            target.sector_key = source.sector_key,
            target.seniority_level = source.seniority_level,
            target.role_category = source.role_category,
            target.role_level = source.role_level,
            target.role_description = source.role_description,
            target.aliases = source.aliases,
            target.is_manager = source.is_manager,
            target.is_executive = source.is_executive,
            target.is_active = source.is_active,
            target.updated_at = source.updated_at,
            target.taxonomy_updated_at = source.taxonomy_updated_at
        WHEN NOT MATCHED THEN INSERT *
        """
        
        spark.sql(merge_sql)
        
        # Count metrics
        roles_inserted = spark.sql(f"""
            SELECT COUNT(*) as cnt FROM roles_to_merge
            WHERE canonical_role_id NOT IN (SELECT canonical_role_id FROM {TARGET_TABLE})
        """).collect()[0]['cnt']
        
        roles_updated = roles_count - roles_inserted
        
        print(f"✓ Merge complete: {roles_inserted} new, {roles_updated} updated")
    
    # Log to metadata
    metadata_data = [(
        run_id,
        roles_count,
        roles_inserted,
        roles_updated,
        FORCE_FULL_REFRESH,
        run_timestamp,
        'success',
        None
    )]
    
    metadata_record = spark.createDataFrame(metadata_data, schema=metadata_schema)
    metadata_record.write.format("delta").mode("append").saveAsTable(METADATA_TABLE)
    
    result = {
        "status": "success",
        "run_id": run_id,
        "roles_extracted": roles_count,
        "roles_inserted": roles_inserted,
        "roles_updated": roles_updated,
        "target_table": TARGET_TABLE,
        "metadata_table": METADATA_TABLE
    }
    
    print(json.dumps(result, indent=2))
    
except Exception as e:
    error_msg = str(e)
    print(f"✗ Error: {error_msg}")
    
    # Log failure to metadata
    metadata_data = [(
        run_id,
        roles_count if 'roles_count' in locals() else 0,
        0,
        0,
        FORCE_FULL_REFRESH,
        run_timestamp,
        'failed',
        error_msg
    )]
    
    metadata_record = spark.createDataFrame(metadata_data, schema=metadata_schema)
    metadata_record.write.format("delta").mode("append").saveAsTable(METADATA_TABLE)
    
    raise

In [0]:
%sql
-- Validate role dimension
SELECT 
  COUNT(*) as total_roles,
  COUNT(DISTINCT role_category) as role_categories,
  COUNT(DISTINCT role_family) as role_families,
  COUNT(DISTINCT sector_key) as sectors,
  COUNT(DISTINCT seniority_level) as seniority_levels,
  SUM(CASE WHEN is_manager THEN 1 ELSE 0 END) as manager_roles,
  SUM(CASE WHEN is_executive THEN 1 ELSE 0 END) as executive_roles,
  SUM(CASE WHEN is_active THEN 1 ELSE 0 END) as active_roles
FROM workspace.warehouse.dim_role;

-- Sample roles by category and seniority
SELECT 
  role_sk,
  role_name,
  family_key,
  role_family,
  sector_key,
  role_category,
  seniority_level,
  role_description,
  is_manager,
  is_executive,
  SIZE(aliases) as alias_count,
  is_active,
  created_at,
  updated_at,
  taxonomy_updated_at
FROM workspace.warehouse.dim_role
ORDER BY role_category, seniority_level, role_name
LIMIT 25;



In [0]:
%sql
-- Show refresh history
SELECT 
  run_id,
  roles_extracted,
  roles_inserted,
  roles_updated,
  force_full_refresh,
  processed_at,
  status
FROM workspace.metadata.dim_role_refresh_log
ORDER BY processed_at DESC
LIMIT 10;